# 03 — Privacy & Anonymization

Detect and remove PII from documents **before** they reach the LLM.

## Why anonymize before extraction?

When you send a document to an LLM for extraction, the full text goes to a remote API. If that document contains personal information — names, email addresses, phone numbers, credit card numbers, national IDs — that PII is now in the hands of a third-party provider.

Even if you trust the provider, regulations like GDPR, HIPAA, and CCPA impose strict rules about *which systems* can see personal data and *how* it must be handled. Sending raw PII to an LLM often violates data minimization principles: the model only needs the *structure* of the data, not the actual personal identifiers.

DocFlow solves this by inserting an **Anonymize** step between Parse and Extract:

```
Ingest → Parse → Anonymize → Extract → Validate → Review → Store
                 ^^^^^^^^^^^
                 PII detected and replaced
                 BEFORE text reaches the LLM
```

The anonymizer scans the parsed text for PII entities using [Microsoft Presidio](https://microsoft.github.io/presidio/), replaces them according to your chosen mode, and passes the cleaned text to the LLM. The model extracts fields from sanitized content — it never sees the real personal data.

### The four anonymization modes

| Mode | What it does | Example | Reversible? |
|------|-------------|---------|-------------|
| **redact** | Replaces PII with `[REDACTED]` | `John Smith` → `[REDACTED]` | No |
| **mask** | First character + asterisks | `John Smith` → `J*********` | No |
| **pseudonymize** | Consistent fake tokens | `John Smith` → `PERSON_001` | Yes |
| **hash** | SHA-256 prefix (16 chars) | `John Smith` → `ef61a579c907bae1` | No |

**Pseudonymize** is the most useful mode for extraction: the LLM sees consistent tokens (`PERSON_001` appears everywhere `John Smith` appeared), so it can still reason about relationships between fields. After extraction, you can reverse the tokens back to real values.

### Fail-closed behavior

When `fail_closed=True` (the default), the pipeline **stops** if anonymization fails — no data is sent to the LLM. This is the safe default for regulated environments: if you can't guarantee PII was removed, don't proceed.

When `fail_closed=False`, a failed anonymization logs a warning and continues with the original text. Use this only in development or when PII leakage is an accepted risk.

### Risk scoring

Each anonymization produces a **risk score** (0–1) based on:
- **PII density** — what fraction of the text is personal data
- **Entity types** — financial entities (credit cards, IBANs) score higher than names
- **Diversity** — documents with many different PII types are riskier

This feeds into the confidence scoring system so downstream consumers know how much PII was in the source document.

In [ ]:
import os

os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "")

PDF_PATH = "data/sample_invoice.pdf"

## Detecting PII with Presidio

Before we anonymize anything, let's see what PII Presidio finds in a parsed document. We'll parse the invoice first, then run the provider's detection on the raw text.

In [ ]:
from docflow.parsing.pdfplumber_parser import PdfplumberParser

parser = PdfplumberParser()
document = parser.parse_sync(PDF_PATH)

print(f"Pages: {len(document.pages)}")
print(f"Raw text length: {len(document.raw_text)} chars")
print(f"\nFirst 500 chars:")
print(document.raw_text[:500])

In [ ]:
import asyncio
from docflow.privacy import PresidioProvider

provider = PresidioProvider(language="en")

# Detect PII in the first page
page_text = document.pages[0].text
findings = asyncio.get_event_loop().run_until_complete(
    provider.adetect_text(page_text, score_threshold=0.35)
)

print(f"Found {len(findings)} PII entities on page 1:\n")
for f in findings:
    print(f"  {f.entity_type:<20} score={f.score:.2f}  \"{f.text}\"")

## Anonymization modes side by side

Let's see how each mode transforms the same text. We'll take a sample sentence with PII and apply all four modes.

In [ ]:
from docflow.privacy import Anonymizer, PrivacyPolicy, AnonymizationMode

sample_text = "Invoice from John Smith (john.smith@meridian.com) for $5,000. Card ending 4242-4242-4242-4242."

for mode in AnonymizationMode:
    policy = PrivacyPolicy(
        provider=PresidioProvider(),
        mode=mode,
        reversible=(mode == AnonymizationMode.PSEUDONYMIZE),
        fail_closed=True,
    )
    anonymizer = Anonymizer(policy)
    result = asyncio.get_event_loop().run_until_complete(
        anonymizer.anonymize_text(sample_text)
    )
    print(f"{mode.value:>14}: {result.text}")

## Pseudonymization in detail

Pseudonymize mode replaces each PII entity with a consistent token like `PERSON_001`. The same original value always maps to the same token within a document, so the LLM can still see patterns ("this person is mentioned 3 times").

Let's anonymize the full document and inspect the mappings.

In [ ]:
policy = PrivacyPolicy(
    provider=PresidioProvider(),
    mode=AnonymizationMode.PSEUDONYMIZE,
    reversible=True,
    fail_closed=True,
)

anonymizer = Anonymizer(policy)
anon_result = asyncio.get_event_loop().run_until_complete(
    anonymizer.anonymize_document(document)
)

print(f"Document ID: {anon_result.document_id}")
print(f"Mode: {anon_result.mode}")
print(f"Risk score: {anon_result.risk_score:.3f}")
print(f"Total findings: {len(anon_result.findings)}")
print(f"Token mappings: {len(anon_result.token_mappings)}")
print(f"Pages processed: {len(anon_result.page_results)}")

In [ ]:
print("Token mappings (pseudonymized → original):\n")
for m in anon_result.token_mappings:
    print(f"  {m.token:<25} → {m.original:<30} ({m.entity_type})")

In [ ]:
print("Anonymized text (first 600 chars):\n")
print(anon_result.anonymized_text[:600])

## Pipeline integration

In practice, you don't call the anonymizer directly — you pass a `PrivacyPolicy` to `DocumentPipeline` and it handles everything. The Anonymize step runs after parsing, before extraction.

The LLM receives the anonymized text and extracts fields from it. The extracted values will contain tokens (`PERSON_001`) instead of real names — which is exactly what we want for the LLM call.

In [ ]:
from docflow import DocumentPipeline, PrivacyPolicy
from docflow.privacy import PresidioProvider, AnonymizationMode
from pydantic import BaseModel, Field


class Invoice(BaseModel):
    supplier_name: str = Field(description="Name of the supplier company")
    invoice_number: str = Field(description="Invoice reference number")
    invoice_date: str = Field(description="Date of the invoice")
    total: float = Field(description="Grand total amount including tax")


pipeline = DocumentPipeline(
    parser="pdfplumber",
    model="gemini/gemini-2.5-flash",
    privacy=PrivacyPolicy(
        provider=PresidioProvider(),
        mode=AnonymizationMode.PSEUDONYMIZE,
        reversible=True,
        fail_closed=True,
    ),
)

result = pipeline.run_sync(PDF_PATH, schema=Invoice)

print("Extracted data (from anonymized text):")
for k, v in result.data.items():
    print(f"  {k:<20}: {v}")
print(f"\nConfidence: {result.confidence:.2f}")
print(f"Needs review: {result.needs_review}")

## Redact mode — maximum privacy

When you need the strongest guarantee that no PII reaches the LLM, use `redact` mode. All detected entities become `[REDACTED]`, with no way to reverse them. The LLM sees the structure but no personal data at all.

In [ ]:
redact_pipeline = DocumentPipeline(
    parser="pdfplumber",
    model="gemini/gemini-2.5-flash",
    privacy=PrivacyPolicy(
        provider=PresidioProvider(),
        mode=AnonymizationMode.REDACT,
        reversible=False,
        fail_closed=True,
    ),
)

result_redacted = redact_pipeline.run_sync(PDF_PATH, schema=Invoice)

print("Extracted with redaction (PII fields may show [REDACTED]):")
for k, v in result_redacted.data.items():
    print(f"  {k:<20}: {v}")

## Controlling which entities to detect

By default, the policy detects common PII types: `PERSON`, `EMAIL_ADDRESS`, `PHONE_NUMBER`, `IBAN_CODE`, `CREDIT_CARD`, `LOCATION`, `DATE_TIME`. You can narrow or widen this list depending on your compliance needs.

In [ ]:
# Detect only names and emails — leave dates and locations alone
narrow_policy = PrivacyPolicy(
    provider=PresidioProvider(),
    mode=AnonymizationMode.PSEUDONYMIZE,
    reversible=True,
    entities=["PERSON", "EMAIL_ADDRESS"],
)

narrow_anonymizer = Anonymizer(narrow_policy)
narrow_result = asyncio.get_event_loop().run_until_complete(
    narrow_anonymizer.anonymize_document(document)
)

print(f"Narrow entity detection:")
print(f"  Findings: {len(narrow_result.findings)} (vs {len(anon_result.findings)} with full detection)")
print(f"  Risk score: {narrow_result.risk_score:.3f} (vs {anon_result.risk_score:.3f})")
print(f"\nEntity types found:")
entity_counts = {}
for f in narrow_result.findings:
    entity_counts[f.entity_type] = entity_counts.get(f.entity_type, 0) + 1
for entity_type, count in sorted(entity_counts.items()):
    print(f"  {entity_type:<20}: {count}")

## Risk score breakdown

The risk score tells you how much PII was in the document. Let's compare the findings across pages.

In [ ]:
print(f"Overall risk score: {anon_result.risk_score:.3f}\n")

print("Findings by entity type:")
entity_summary = {}
for f in anon_result.findings:
    if f.entity_type not in entity_summary:
        entity_summary[f.entity_type] = {"count": 0, "avg_score": 0.0}
    entity_summary[f.entity_type]["count"] += 1
    entity_summary[f.entity_type]["avg_score"] += f.score

for etype, info in sorted(entity_summary.items(), key=lambda x: -x[1]["count"]):
    avg = info["avg_score"] / info["count"]
    print(f"  {etype:<20}: {info['count']:3d} occurrences, avg confidence {avg:.2f}")

print(f"\nFindings per page:")
for i, page_result in enumerate(anon_result.page_results):
    print(f"  Page {i}: {len(page_result.findings)} findings, {len(page_result.mappings)} mappings")

## Fail-closed vs fail-open

When `fail_closed=True`, the pipeline stops if anonymization fails — this is the safe default for production. Let's demonstrate both behaviors with a deliberately broken provider.

In [ ]:
from docflow.privacy.models import PrivacyFinding, TokenMapping


class BrokenProvider:
    """A provider that always fails — for demonstrating fail_closed."""
    async def adetect_text(self, text, **kwargs):
        raise RuntimeError("Provider unavailable")

    async def aanonymize_text(self, text, findings, mode, **kwargs):
        return text, []

    async def arestore_text(self, text, mappings):
        return text


# fail_closed=True — pipeline stops
try:
    fail_closed_policy = PrivacyPolicy(
        provider=BrokenProvider(),
        mode=AnonymizationMode.REDACT,
        reversible=False,
        fail_closed=True,
    )
    anon = Anonymizer(fail_closed_policy)
    asyncio.get_event_loop().run_until_complete(anon.anonymize_text("John Smith"))
except Exception as e:
    print(f"fail_closed=True  → {type(e).__name__}: {e}")

# fail_closed=False — continues with original text
fail_open_policy = PrivacyPolicy(
    provider=BrokenProvider(),
    mode=AnonymizationMode.REDACT,
    reversible=False,
    fail_closed=False,
)
anon_open = Anonymizer(fail_open_policy)
result_open = asyncio.get_event_loop().run_until_complete(
    anon_open.anonymize_text("John Smith")
)
print(f"fail_closed=False → text passes through: \"{result_open.text}\"")

## Reversing pseudonymization

When using pseudonymize mode with `reversible=True` and a mapping store, you can restore tokens back to real values after extraction. This gives you the best of both worlds: the LLM never sees PII, but the final output has the real data.

In [ ]:
from docflow.privacy.mapping_store import LocalMappingStore
import tempfile

store_dir = tempfile.mkdtemp()
mapping_store = LocalMappingStore(store_dir)

restore_policy = PrivacyPolicy(
    provider=PresidioProvider(),
    mode=AnonymizationMode.PSEUDONYMIZE,
    reversible=True,
    mapping_store=mapping_store,
)

restore_anonymizer = Anonymizer(restore_policy)

# Anonymize
sample = "Please contact John Smith at john.smith@meridian.com regarding invoice #12345."
anon = asyncio.get_event_loop().run_until_complete(
    restore_anonymizer.anonymize_text(sample)
)
print(f"Original:    {sample}")
print(f"Anonymized:  {anon.text}")
print(f"Mapping ID:  {anon.mapping_id}")

# Restore
restored = asyncio.get_event_loop().run_until_complete(
    restore_anonymizer.restore_text(anon.text, anon.mapping_id)
)
print(f"Restored:    {restored}")
print(f"\nRound-trip match: {restored == sample}")

## Tuning detection sensitivity

The `score_threshold` parameter controls how aggressive PII detection is. Lower values catch more potential PII (including false positives); higher values only flag high-confidence detections.

In [ ]:
test_text = document.pages[0].text

for threshold in [0.2, 0.35, 0.5, 0.7, 0.85]:
    findings = asyncio.get_event_loop().run_until_complete(
        provider.adetect_text(test_text, score_threshold=threshold)
    )
    entity_types = set(f.entity_type for f in findings)
    print(f"  threshold={threshold:.2f}: {len(findings):3d} findings ({', '.join(sorted(entity_types)) or 'none'})")

## Summary

| Feature | Description |
|---------|-------------|
| **Pipeline slot** | Between Parse and Extract — the LLM never sees raw PII |
| **Provider** | Microsoft Presidio (pluggable via `PrivacyProvider` protocol) |
| **Modes** | `redact`, `mask`, `pseudonymize`, `hash` |
| **Reversible** | Pseudonymize mode + mapping store → restore after extraction |
| **Fail-closed** | Default: pipeline stops if anonymization fails |
| **Risk score** | 0–1 based on PII density, entity types, and diversity |
| **Entity control** | Configure which PII types to detect |
| **Threshold** | Tune detection sensitivity (default 0.35) |

```python
# The one-line version:
pipeline = DocumentPipeline(
    parser="pdfplumber",
    model="gemini/gemini-2.5-flash",
    privacy=PrivacyPolicy(
        provider=PresidioProvider(),
        mode="pseudonymize",
        reversible=True,
    ),
)
```